# GitHub Repository RAG

### Retrieval-Augmented Generation (RAG) system for querying and interacting with the contents of GitHub repositories.

---

**Author:** David Fernández Martínez  
**Email:** [david.fernxndez.martinez@gmail.com](mailto:david.fernxndez.martinez@gmail.com)  
**LinkedIn:** [linkedin.com/in/david-fernández-martínez](https://linkedin.com/in/david-fernández-martínez)  
**GitHub:** [github.com/davidfernxndez](https://github.com/davidfernxndez)

---

## 08 · Prompt and LLM Generation

This notebook presents the **two final components of the RAG pipeline: prompt construction and LLM generation**.

First, a prompt is constructed using the document chunks retrieved by the retrieval component. The prompt provides the LLM with the relevant context required to answer the user's question, while also defining instructions that guide how the answer should be generated.

Second, the resulting prompt is passed to an **Large Language Model (LLM)**, which uses the provided context to generate the final response. The system supports two different approaches for interacting with the LLM. For local execution, it can use a model served through **Ollama**, allowing the complete RAG pipeline to run locally on a computer without requiring a GPU. For the Streamlit deployment, the system uses the **Groq API**, providing access to a remote LLM through an API without requiring the model to be hosted by the application.

This separation between prompt construction and LLM generation keeps the RAG components independent and allows the underlying LLM to be changed without modifying the retrieval or prompt-generation logic.

**Notebook objectives**

* Present the prompt template used to provide the LLM with the retrieved context.
* Explain and justify the main design decisions behind the prompt.
* Present the characteristics of the selected LLMs for local and deployed execution.
* Generate responses using the same prompt with both LLM configurations.

## Imports and Configuration

In [2]:
import time
from pathlib import Path
from IPython.display import Markdown, display
from github_rag.ingestion.repository import GitHubRepositoryLoader
from github_rag.processing.processor import Processor
from github_rag.chunking.chunker import Chunker
from github_rag.embedding.embedding_model import create_embedding_model
from github_rag.vectordatabase.chroma_store import ChromaStore
from github_rag.retrieval.retriever import Retriever
from github_rag.prompt_engineering.prompt_builder import build_prompt
from github_rag.LLM_generation.generator import LLMGenerator

c:\Users\david\anaconda3\envs\github-rag\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


You can explore any GitHub repository simply by providing its URL. For this demo, I'll be using one of my own end-to-end Data Science projects that implements and evaluates a distributed Active Learning solution in PySpark.

In [3]:
REPOSITORY_URL = "https://github.com/davidfernxndez/spark-active-learning"

## 1. Pipeline Setup

Run the complete RAG pipeline, ensuring that all the components are properly initialized and ready to use.

In [4]:
# Set the Loader Object with the URL
loader = GitHubRepositoryLoader(REPOSITORY_URL)

# Load the repository files as a list of LangChain Documents
documents = loader.load()

print(f"Number of documents ingested: {len(documents)}")

# Set the Processor Object
processor = Processor()

# Apply processing to the ingested documents
documents_processed = processor.process(documents)

print(f"Number of documents after processing: {len(documents_processed)}")

# Set the Chunker object that managed all the specific chunkers
chunker = Chunker()

# Apply chunking
documents_chunking = chunker.chunk(documents_processed)

print(f"Number of documents after chunking: {len(documents_chunking)}")

# Load the HuggingFace embedding model
start_time = time.time()
embedding_model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
embedding_model = create_embedding_model(embedding_model_name)
print(f"Embedding model loaded time: {time.time() - start_time:.2f} seconds")

# Create the vector database storing
start_time = time.time()
vector_store = ChromaStore(
    mode="local",
    collection_name="github_repo"
)

# Create the database with documents and the embedding model
vector_store.create(
    documents=documents_chunking,
    embedding_model=embedding_model
)
print(f"Database generation time: {time.time() - start_time:.2f} seconds")

# Check the number of stored documents
stored_documents = vector_store.count()

print(f"Documents provided: {len(documents_chunking)}")
print(f"Documents stored:   {stored_documents}")

# Get the available files for the Retriever component
available_files = vector_store.get_unique_file_paths()
print(f"Number of available files: {len(available_files)}")

# Create the Retriever object
retriever = Retriever(vector_store)

Number of documents ingested: 12
Number of documents after processing: 45
Number of documents after chunking: 183
Embedding model loaded time: 17.87 seconds
Database generation time: 7.26 seconds
Documents provided: 183
Documents stored:   183
Number of available files: 12


A example query is defined to understand the prompt design and to use the LLMs models.

In [5]:
query = "How is the instance closest to each cluster centroid selected?"

In [7]:
# Retrieve the relevant chunks
relevant_chunks = retriever.retrieve(query, available_files)

## 2. Prompt Template Design

The prompt is responsible for providing the LLM with the information and instructions required to generate an accurate answer. Its design follows a simple structure that separates the **retrieved context**, the **user's question**, and the **generation instructions**.

The retrieved context is constructed from the chunks returned by the retrieval component. For each chunk, both its textual content and relevant metadata are included:

* **File name:** identifies the source file from which the chunk was retrieved.
* **Source:** provides the URL of the original file in the GitHub repository.
* **Content:** contains the actual text retrieved from the repository.

Although the metadata is not included when generating the embeddings, it is preserved in the vector database and added to the prompt at this stage. This allows the LLM to use the source information when generating citations for its answer.

The prompt provides the following instructions to the LLM:

* **Ground the answer in the retrieved context:** the model is instructed to use the retrieved information as the basis for its response.
* **Avoid hallucinations:** when the retrieved context does not contain enough information to answer the question, the model is explicitly instructed to acknowledge this rather than guessing.
* **Provide source citations:** the model must identify the files used to answer the question.
* **Use a consistent citation format:** citations are formatted as Markdown links, using the file name as the link text and the GitHub source URL as the destination.

Below, the prompt for the example query is shown.

In [10]:
# Build the prompt using the retrieved chunks
prompt = build_prompt(relevant_chunks, query)
print(prompt)


    You are an assistant specialized in answering questions
    about GitHub repositories.

    Use the retrieved context to answer the user's question.

    Context:
    File_name: main.ipynb
    Source: https://github.com/davidfernxndez/spark-active-learning/blob/main/main.ipynb

    Content:
    **Selecting the Instance Closest to Each Cluster Centroid**  
The goal of this stage is to select a single representative instance from each cluster, specifically the one with the minimum distance to its centroid. To achieve this, the $B$ centroids are distributed using *broadcast*, allowing each *worker* to access them locally while processing the instances.  
Since candidates belonging to the same cluster may be distributed across different partitions, all partitions need access to the same centroids. Using *broadcast* makes this information locally available and avoids repeatedly redistributing it during processing.  
The distance between each instance and the centroid of its assigned cl

## 3. LLM Configuration

The `LLMGenerator` class supports two different execution modes: a **local LLM running through Ollama** and an **API-based LLM accessed through Groq**. This provides flexibility to run the complete RAG system locally or use a more powerful model for the deployed Streamlit application.

#### 3.1 Local LLM with Ollama

For local execution, the system uses **Ollama** to run the LLM locally. Ollama must be installed and the selected model must be available in the local environment. The selected model is `qwen3:1.7b`.

This model was selected as a compromise between **computational requirements and response quality**. With approximately 2 billion parameters and a quantized model size of around 1.4 GB, it is small enough to run on a relatively modest computer, including CPU-only environments, while still providing useful instruction-following and reasoning capabilities. The Qwen3 family also supports multilingual interaction and reasoning capabilities.

This configuration is particularly useful for development and experimentation because the entire RAG pipeline can be executed locally without requiring access to an external LLM API.

#### 3.2 API-based LLM with Groq

For the deployed Streamlit application, the system can use the **Groq API** instead of running the model locally. This avoids the computational and memory requirements of hosting an LLM on the deployment server. The selected model is `openai/gpt-oss-20b`.

This is a substantially more capable model than the local 1.7B model. It is an open-weight **Mixture-of-Experts (MoE)** model with 20 billion total parameters, of which approximately 3.6 billion are active for each forward pass. It supports reasoning, code generation, tool use, and long-context processing, with a context window of up to 131,072 tokens. Groq provides very fast inference for this model, making it well suited to an interactive RAG application.

To use the Groq configuration, an API key must be provided through the `GROQ_API_KEY` environment variable. The key is loaded from a *.env* file, as described in the *.env.example* file included in the repository. The API is subject to usage and rate limits depending on the account or plan, but these limits are sufficient for normal development and demonstration use.

In [11]:
# Create two LLMGenerator instances
local_LLM = LLMGenerator(mode="local")
API_LLM = LLMGenerator(mode="API")

We can use both models to answer the example query and compare the responses.

In [12]:
# Local model (qwen3:1.7b)
response = local_LLM.generate(prompt)
display(Markdown(response))

The instance closest to each cluster centroid is selected by computing the squared distance between each candidate instance and the corresponding cluster centroid using the broadcasted centroids. This is done locally on each worker after clustering. The process involves:  
1. Transforming the clustered candidates into an RDD where each entry contains the cluster ID and the sample ID along with the squared distance to the centroid.  
2. Using `centers_broadcast.value` to access the centroids locally, avoiding repeated redistribution.  
3. For each cluster, the instance with the minimum distance (i.e., the closest instance to the centroid) is selected.  

This approach ensures efficient computation by leveraging Spark's broadcast mechanism to avoid data shuffling.  

Source:  
Source: [scripts/AL_methods.py](scripts/AL_methods.py)  
Source: [main.ipynb](main.ipynb)

In [13]:
# API model (openai/gpt-oss-20b)
response = API_LLM.generate(prompt)
display(Markdown(response))

The pipeline first clusters the candidate instances with K‑Means.  
For every instance it then computes the squared Euclidean distance between its feature vector and the centroid of the cluster to which it was assigned.  
These distances are paired with the instance’s ID and the cluster ID, producing an RDD of the form  

```
(cluster_id, (id_sample, squared_distance))
```

The centroids are broadcast to every worker (`centers_broadcast.value`) so that the distance calculation can be performed locally without shuffling the data.  
Finally, the RDD is reduced by `cluster_id` keeping only the pair with the smallest distance for each cluster.  
The resulting `id_sample` for each cluster is the instance that is closest to that cluster’s centroid and is selected for labeling.

**Sources**

- [main.ipynb](https://github.com/davidfernxndez/spark-active-learning/blob/main/main.ipynb)  
- [scripts/AL_methods.py](https://github.com/davidfernxndez/spark-active-learning/blob/main/scripts/AL_methods.py)

We can observe that both models successfully answer the user's question. However, the response from openai/gpt-oss-20b offers more detail and a more visual format for the citations.